
## Denormalization:

- **Denormalization** is the process of `combining data from multiple normalized tables into a single, wider table to reduce joins`. 
- It improves a **query performance** for `analytics and reporting`.

**Real-time example: E-commerce sales analytics**

**Suppose an online shopping company stores its data in a star schema.**


### A. Source Tables:

**1. Sales Fact Table**

| SaleID | CustomerID | ProductID | Quantity | Revenue |
|---|---|---|---|---|
| 1 | 101 | 201 | 2 | 2000 |
| 2 | 102 | 202 | 1 | 1500 |
| 3 | 101 | 202 | 3 | 4500 |

**2. Customer Dimension Table**

| CustomerID | CustomerName | City |
|---|---|---|
| 101 | Arun | Chennai |
| 102 | Bala | Madurai |

**3. Product Dimension Table**

| ProductID | ProductName | Category |
|---|---|---|
| 201 | Laptop | Electronics |
| 202 | Shoes | Fashion |




## Create the dataframes:

In [0]:
sales_fact_df = spark.createDataFrame([
    (1, 101, 201, 2, 2000.0),
    (2, 102, 202, 1, 1500.0),
    (3, 101, 202, 3, 4500.0)
], ["SaleID", "CustomerID", "ProductID", "Quantity", "Revenue"])

customer_dim_df = spark.createDataFrame([
    (101, "Arun", "Chennai"),
    (102, "Bala", "Madurai")
], ["CustomerID", "CustomerName", "City"])

product_dim_df = spark.createDataFrame([
    (201, "Laptop", "Electronics"),
    (202, "Shoes", "Fashion")
], ["ProductID", "ProductName", "Category"])

## Denormalize using joins:

In [0]:
wide_sales_df = (
    sales_fact_df
    .join(customer_dim_df, "CustomerID", "left")
    .join(product_dim_df, "ProductID", "left")
)

wide_sales_df.show()


## Perform analytics on the wide dataset:

**Suppose the business wants revenue by city and product category.**

In [0]:
from pyspark.sql import functions as F

wide_sales_df.groupBy("City", "Category").agg(
    F.sum("Revenue").alias("TotalRevenue")
).show()


## Save the denormalized dataset:

In [0]:
wide_sales_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/data_volume"
)


## Review the Output:

In [0]:
wide_sales_df.show()


## Another Real-Time Example: Hospital Management


**A hospital stores patient details, doctor details, and appointment details in separate tables. The data engineering team wants to create a single wide dataset for hospital management dashboards.**

**Table 1: Patients**

| patient_id | patient_name | city |
|---|---|---|
| 101 | Arun | Chennai |
| 102 | Priya | Madurai |
| 103 | Kumar | Theni |

**Table 2: Appointments**

| appointment_id | patient_id | doctor_id | bill_amount |
|---|---|---|---|
| 5001 | 101 | 201 | 1500 |
| 5002 | 102 | 202 | 2000 |
| 5003 | 101 | 202 | 1000 |

**Table 3: Doctors**

| doctor_id | doctor_name | specialization |
|---|---|---|
| 201 | Dr. Ravi | Cardiology |
| 202 | Dr. Meena | Neurology |


## Create the Patients DataFrame:

In [0]:
from pyspark.sql import functions as F

patients_data = [
    (101, "Arun", "Chennai"),
    (102, "Priya", "Madurai"),
    (103, "Kumar", "Theni")
]

columns = ["patient_id", "patient_name", "city"]

patients_df = spark.createDataFrame(
    patients_data,columns 
)

patients_df.show()


## Create the Appointments DataFrame:

In [0]:
appointments_data = [
    (5001, 101, 201, 1500),
    (5002, 102, 202, 2000),
    (5003, 101, 202, 1000)
]

appointments_df = spark.createDataFrame(
    appointments_data,
    ["appointment_id", "patient_id", "doctor_id", "bill_amount"]
)

appointments_df.show()


## Create the Doctors DataFrame:

In [0]:
doctors_data = [
    (201, "Dr. Ravi", "Cardiology"),
    (202, "Dr. Meena", "Neurology")
]

doctors_df = spark.createDataFrame(
    doctors_data,
    ["doctor_id", "doctor_name", "specialization"]
)

doctors_df.show()


## Perform denormalization:

Combine all three tables into a single wide dataset.

In [0]:
hospital_wide_df = (
    appointments_df
    .join(patients_df, on="patient_id", how="inner")
    .join(doctors_df, on="doctor_id", how="inner")
    .select(
        "appointment_id",
        "patient_id",
        "patient_name",
        "city",
        "doctor_id",
        "doctor_name",
        "specialization",
        "bill_amount"
    )
)

display(hospital_wide_df)


## Generate the billing report:

**Calculate the total billing amount for each medical specialization.**

In [0]:
specialization_report_df = hospital_wide_df.groupBy(
    "specialization"
).agg(
    F.sum("bill_amount").alias("total_billing")
)

specialization_report_df.show()